In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.graph_objects as go

from IPython.display import HTML, display

In [ ]:
data_dir = Path("/Users/claudy/dev/work/data/vs30/maurer")

In [ ]:
usgs_df = pd.read_csv(data_dir / "USGS VS30 compilation 2020.csv", index_col=0).rename(columns={
    "LATITUDE": "lat",
    "LONGITUDE": "lon",
    "DATUM": "datum",
    "NETWORK_ST": "network",
    "STATION_NA": "station_name",
    "METHOD": "method",
    "VS30__M_S_": "vs30",
    "MAX_DEPTH": "max_depth",
}).drop(columns=["CONTACT", "REFERENCE", "URL", "GEOLOGIC_D", "COMMENTS"])

In [ ]:
us_df = pd.read_excel(data_dir / "Residuals - Stacked - 6966plus23 v4.xlsx")
us_df = us_df.drop(columns=[col for col in us_df.columns if col not in ["Latitude", "Longitude", "Observation (m/s)"]]).rename(columns={
    "Latitude": "lat",
    "Longitude": "lon",
    "Observation (m/s)": "vs30",
})

# Save 
#us_df.to_csv("/Users/claudy/dev/work/data/vs30/us/geyin_maurer_vs30.csv", index=False)

## Spatial Distribution

In [ ]:
fig = go.Figure()

fig.update_layout(width=1400, height=800,
                  margin=dict(l=10, r=10, t=30, b=10),
                  map=dict(zoom=3, center=dict(lat=us_df.lat.mean(), lon=us_df.lon.mean())),
                #   showlegend=True
)

# USGS
fig.add_trace(go.Scattermap(
    lon = usgs_df['lon'],
    lat = usgs_df['lat'],
    name='USGS VS30 Compilation',
    mode = 'markers',
    marker = dict(
        size = 4,
        color="red")
))

# US
fig.add_trace(go.Scattermap(
    lon = us_df['lon'],
    lat = us_df['lat'],
    name='US Measurements',
    mode = 'markers',
    marker = dict(
        size = 4,
        color="blue")
))


## VS30 Distribution

In [ ]:
display(HTML(us_df["vs30"].to_frame().describe().to_html()))

**There are clearly some outliers at the extremes, this will need to addressed prior to model training!**

In [ ]:
bins = np.linspace(0, 2000, 50)

fig, ax = plt.subplots(figsize=(12, 8))

ax.hist(us_df['vs30'], bins=bins, edgecolor='black')

ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlabel("Vs30 (m/s)")
ax.set_ylabel("Count")
ax.set_xlim(bins.min(), bins.max())

fig.tight_layout()